In [5]:
import os
os.environ['http_proxy'] = 'http://127.0.0.1:7890'
os.environ['https_proxy'] = 'http://127.0.0.1:7890'

In [2]:
import torch
import torch.nn.functional as F
from copy import deepcopy

$$
 \dfrac{x_1 \cdot x_2}{\max(\Vert x_1 \Vert _2 \cdot \Vert x_2 \Vert _2, \epsilon)}
$$

In [6]:
F.cosine_similarity??

In [3]:
from sentence_transformers import SentenceTransformer, LoggingHandler, losses, InputExample, models
from torch.utils.data import DataLoader

model = SentenceTransformer('all-MiniLM-L6-v2', device='cpu')

[2024-02-10 19:40:51,465] [INFO] [real_accelerator.py:161:get_accelerator] Setting ds_accelerator to cuda (auto detect)


/home/whaow/anaconda3/lib/python3.10/site-packages/torch/_utils.py:776: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  return self.fget.__get__(instance, owner)()


## model

In [9]:
model

SentenceTransformer(
  (0): Transformer({'max_seq_length': 256, 'do_lower_case': False}) with Transformer model: BertModel 
  (1): Pooling({'word_embedding_dimension': 384, 'pooling_mode_cls_token': False, 'pooling_mode_mean_tokens': True, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False})
  (2): Normalize()
)

- `model[0]`: `token_embeddings` 
- `model[1]`: `pooling_mode_mean_tokens`
    - `sentence_embedding`: 

- 其他的 Pooling 方法

```
pooling_model = models.Pooling(word_embed_model.get_word_embedding_dimension(), 
                       pooling_mode='cls', # classifiy
                       pooling_mode_cls_token=True, 
                       pooling_mode_mean_tokens = False)
```

## dataloader

In [8]:
train_examples = [
    InputExample(texts=['This is a positive pair', 
                        'Where the distance will be minimized'], 
                 label=1),
#     InputExample(texts=['This is a negative pair', 'Their distance will be increased'], label=0)
]

In [9]:
# sentences input, 
train_dataloader = DataLoader(train_examples, shuffle=True, batch_size=2)

In [10]:
train_dataloader.collate_fn

<function torch.utils.data._utils.collate.default_collate(batch)>

In [12]:
train_dataloader.collate_fn = model.smart_batching_collate
batch = next(iter(train_dataloader))
batch 

([{'input_ids': tensor([[ 101, 2023, 2003, 1037, 3893, 3940,  102]]),
   'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0]]),
   'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1]])},
  {'input_ids': tensor([[  101,  2073,  1996,  3292,  2097,  2022, 18478,  2094,   102]]),
   'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0]]),
   'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1]])}],
 tensor([1]))

## losses

- $(a,b)$: pair sentences embeddings

$$
\frac12|a,b|^2, \ell=1\\
\text{ReLU}^2(\epsilon-|a,b|), \ell=0
$$

In [13]:
losses.ContrastiveLoss??

In [14]:
train_loss = losses.ContrastiveLoss(model=model)

In [18]:
# list(train_loss.named_parameters())

In [15]:
list(train_loss.named_parameters())[0][1].shape

torch.Size([30522, 384])

In [21]:
# SiameseDistanceMetric.COSINE_DISTANCE
# lambda x, y: 1-F.cosine_similarity(x, y)
train_loss.distance_metric

<function sentence_transformers.losses.ContrastiveLoss.SiameseDistanceMetric.<lambda>(x, y)>

## model.forward

In [23]:
batch

([{'input_ids': tensor([[ 101, 2023, 2003, 1037, 3893, 3940,  102]]),
   'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0]]),
   'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1]])},
  {'input_ids': tensor([[  101,  2073,  1996,  3292,  2097,  2022, 18478,  2094,   102]]),
   'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0]]),
   'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1]])}],
 tensor([1]))

In [22]:
batch[1]

tensor([1])

In [18]:
batch[0][0]['input_ids'].shape

torch.Size([1, 7])

In [24]:
features, labels = batch
features

[{'input_ids': tensor([[ 101, 2023, 2003, 1037, 3893, 3940,  102]]),
  'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0]]),
  'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1]])},
 {'input_ids': tensor([[  101,  2073,  1996,  3292,  2097,  2022, 18478,  2094,   102]]),
  'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0]]),
  'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1]])}]

In [61]:
# model[2](model[1](model[0](features[0])))

In [25]:
model

SentenceTransformer(
  (0): Transformer({'max_seq_length': 256, 'do_lower_case': False}) with Transformer model: BertModel 
  (1): Pooling({'word_embedding_dimension': 384, 'pooling_mode_cls_token': False, 'pooling_mode_mean_tokens': True, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False})
  (2): Normalize()
)

In [26]:
feature_cpy = deepcopy(features)

In [ ]:
model[1](model[0](features[0]))['sentence_embedding'][0, :5] # pooling(bert(input))

tensor([-0.2930,  0.3243, -0.6169, -0.0097, -0.1806], grad_fn=<SliceBackward0>)

In [ ]:
(torch.sum(model[0](feature_cpy[0])['token_embeddings'], dim=1) / 7)[0, :5] # bert得到embedding后取平均，token embedding平均得到sentence embedding

tensor([-0.2930,  0.3243, -0.6169, -0.0097, -0.1806], grad_fn=<SliceBackward0>)

## forward loss

In [28]:
sent1_embed = model(features[0])['sentence_embedding']
sent2_embed = model(features[1])['sentence_embedding']

In [29]:
# 1 - F.cosine_similarity(sent1_embed, sent2_embed)
train_loss.distance_metric(sent1_embed, sent2_embed)

tensor([0.9867], grad_fn=<RsubBackward1>)

In [30]:
1 - F.cosine_similarity(sent1_embed, sent2_embed)

tensor([0.9867], grad_fn=<RsubBackward1>)

In [31]:
train_loss(features, labels)

tensor(0.4868, grad_fn=<MeanBackward0>)

In [27]:
1/2*(1-F.cosine_similarity(sent1_embed, sent2_embed))**2

tensor([0.4868], grad_fn=<MulBackward0>)

In [ ]:
model.fit([(train_dataloader, train_loss)], show_progress_bar=True)

## Pooling methods

### pooling_mode_mean_tokens

In [32]:
train_examples = [
    InputExample(texts=['This is a positive pair', 'Where the distance will be minimized'], label=1),
#     InputExample(texts=['This is a negative pair', 'Their distance will be increased'], label=0)
]
# sentences input, 
train_dataloader = DataLoader(train_examples, shuffle=True, batch_size=2)
train_dataloader.collate_fn = model.smart_batching_collate
batch = next(iter(train_dataloader))
# batch 
features, labels = batch
features

[{'input_ids': tensor([[ 101, 2023, 2003, 1037, 3893, 3940,  102]]),
  'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0]]),
  'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1]])},
 {'input_ids': tensor([[  101,  2073,  1996,  3292,  2097,  2022, 18478,  2094,   102]]),
  'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0]]),
  'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1]])}]

In [33]:
word_embed_model = models.Transformer('bert-base-uncased')
# a pool function over the token embeddings
pooling_model = models.Pooling(word_embed_model.get_word_embedding_dimension(), 
                               pooling_mode_cls_token=False, 
                               pooling_mode_mean_tokens=True, 
                               pooling_mode_max_tokens=False, 
                               pooling_mode_mean_sqrt_len_tokens=False)
model = SentenceTransformer(modules=[word_embed_model, pooling_model])

In [46]:
model(features[0])['sentence_embedding'][0][:5]

tensor([-0.0863, -0.2668,  0.5492, -0.4936, -0.1374], grad_fn=<SliceBackward0>)

### cls pooling method

In [47]:
train_examples = [
    InputExample(texts=['This is a positive pair', 'Where the distance will be minimized'], label=1),
#     InputExample(texts=['This is a negative pair', 'Their distance will be increased'], label=0)
]
# sentences input, 
train_dataloader = DataLoader(train_examples, shuffle=True, batch_size=2)
train_dataloader.collate_fn = model.smart_batching_collate
batch = next(iter(train_dataloader))
# batch 
features, labels = batch
features

[{'input_ids': tensor([[ 101, 2023, 2003, 1037, 3893, 3940,  102]]),
  'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0]]),
  'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1]])},
 {'input_ids': tensor([[  101,  2073,  1996,  3292,  2097,  2022, 18478,  2094,   102]]),
  'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0]]),
  'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1]])}]

In [48]:
word_embed_model = models.Transformer('bert-base-uncased')
# a pool function over the token embeddings
pooling_model = models.Pooling(word_embed_model.get_word_embedding_dimension(), 
                               pooling_mode = 'cls',
                               pooling_mode_cls_token=True, 
                               pooling_mode_mean_tokens = False)
model = SentenceTransformer(modules=[word_embed_model, pooling_model])

In [49]:
model(features[0])['sentence_embedding'][0][:5]

tensor([-0.1775, -0.0474,  0.1351, -0.3242, -0.5006], grad_fn=<SliceBackward0>)

### cls pooling from scartch

In [50]:
train_examples = [
    InputExample(texts=['This is a positive pair', 'Where the distance will be minimized'], label=1),
#     InputExample(texts=['This is a negative pair', 'Their distance will be increased'], label=0)
]
# sentences input, 
train_dataloader = DataLoader(train_examples, shuffle=True, batch_size=2)
train_dataloader.collate_fn = model.smart_batching_collate
batch = next(iter(train_dataloader))
# batch 
features, labels = batch
features

[{'input_ids': tensor([[ 101, 2023, 2003, 1037, 3893, 3940,  102]]),
  'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0]]),
  'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1]])},
 {'input_ids': tensor([[  101,  2073,  1996,  3292,  2097,  2022, 18478,  2094,   102]]),
  'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0]]),
  'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1]])}]

In [ ]:

model[0](features[0])['token_embeddings'].shape

torch.Size([1, 7, 768])

In [56]:
model[0](features[0])['token_embeddings'][0, 0][:5]

tensor([-0.1775, -0.0474,  0.1351, -0.3242, -0.5006], grad_fn=<SliceBackward0>)

你想通过具体的例子搞懂「CLS池化如何把一堆token级的embedding变成一个句子级的embedding」，这个思路特别好！我用**最通俗的语言+可视化例子**给你讲透，保证一看就懂。

### 先铺垫：先明确3个核心概念（例子的基础）
为了简化，我把模型的隐藏层维度设为`4维`（实际bert-base是768维，原理完全一样），文本也选最简单的：
- 输入句子：`我 爱 编程`
- 模型处理后会给句子加特殊token，最终token序列：`[CLS] 我 爱 编程 [SEP]`（共5个token）
- 每个token都会被模型编码成一个**4维的向量（embedding）**，比如：
  - `[CLS]` → [0.2, 0.5, 0.1, 0.8]
  - `我` → [0.3, 0.4, 0.7, 0.2]
  - `爱` → [0.6, 0.1, 0.9, 0.4]
  - `编程` → [0.8, 0.3, 0.2, 0.7]
  - `[SEP]` → [0.1, 0.9, 0.5, 0.3]

### 例子：CLS池化的完整过程（一步到位）
#### 第一步：模型输出token级embedding矩阵
模型处理完句子后，会输出一个「token×维度」的矩阵（形状：5个token × 4维），像这样：

| token    | 维度1 | 维度2 | 维度3 | 维度4 |
|----------|-------|-------|-------|-------|
| [CLS]    | 0.2   | 0.5   | 0.1   | 0.8   |
| 我       | 0.3   | 0.4   | 0.7   | 0.2   |
| 爱       | 0.6   | 0.1   | 0.9   | 0.4   |
| 编程     | 0.8   | 0.3   | 0.2   | 0.7   |
| [SEP]    | 0.1   | 0.9   | 0.5   | 0.3   |

这个矩阵就是「token级embedding」——每个token都有自己的向量，但我们需要的是**整个句子的一个向量**。

#### 第二步：CLS池化的核心操作（只取第一行！）
CLS池化的本质就是：**直接提取`[CLS]`对应的那一行向量，作为整个句子的embedding**。

从上面的矩阵中，只拿走第一行（`[CLS]`的向量）：
```
句子级embedding = [0.2, 0.5, 0.1, 0.8]
```

#### 第三步：（可选）L2归一化（SBERT默认做）
为了让向量更适合相似度计算，会把这个4维向量做「归一化」（让向量长度变成1）：
- 计算原向量的长度：√(0.2² + 0.5² + 0.1² + 0.8²) = √(0.04+0.25+0.01+0.64) = √0.94 ≈ 0.97
- 归一化后：[0.2/0.97, 0.5/0.97, 0.1/0.97, 0.8/0.97] ≈ [0.206, 0.515, 0.103, 0.825]

最终，这个**归一化后的4维向量**，就是CLS池化得到的「句子级embedding」。

### 关键补充：为什么`[CLS]`能代表整个句子？
你可能会问：为什么只拿第一个token的向量，就能代表整句话？
这是BERT模型的设计逻辑决定的：
- 预训练时，模型被要求用`[CLS]`向量完成「句子分类」「下一句预测」等任务；
- 模型在学习过程中，会自动把整句话的**全局语义信息**“浓缩”到`[CLS]`向量里——就像用一个“总结句”代表整篇文章，`[CLS]`就是模型给句子做的“语义总结”。

### 对比理解（更清楚CLS的特点）
如果用其他池化方式（比如mean平均池化），会把所有token的向量加起来求平均：
- mean池化结果 = [(0.2+0.3+0.6+0.8+0.1)/5, (0.5+0.4+0.1+0.3+0.9)/5, ...] = [0.4, 0.44, 0.48, 0.48]
- 而CLS池化是“直接拿总结向量”，不是“平均所有向量”——这是核心区别。

### 总结
CLS池化把token embedding变成句子embedding的过程，用一句话概括就是：
1. 模型给每个token生成专属向量，其中`[CLS]`向量是模型对整句话的“语义总结”；
2. CLS池化直接提取这个“总结向量”，（可选）归一化后，就得到了句子级embedding；
3. 整个过程没有复杂计算，核心是“取第一个token的向量”，简单且符合BERT的设计初衷。

### 关键点回顾
- CLS池化的操作：**只提取`[CLS]`对应的token向量**，一步完成token级→句子级的转换；
- `[CLS]`的特殊性：模型预训练时已学会将句子全局语义浓缩到这个token中；
- 与mean池化的区别：CLS是“取总结向量”，mean是“平均所有向量”。